In [1]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
import mysql.connector

print("MySQL Connector/Python: installed successfully")

MySQL Connector/Python: installed successfully


In [5]:
# ============================================================
# PHASE 6 — MYSQL DATA ARCHITECTURE & LOADING
# ============================================================
#
# Architecture:
#
#       Phase 5 Frozen CSV
#                |
#                v
#              RAW
#                |
#                v
#            STAGING
#                |
#                v
#              CORE
#          /     |      \
#      customer supplier product
#                |
#                v
#            ANALYTICS
#
# Plus:
#            METADATA
#          pipeline_runs
#
# This phase does NOT redo Pandas cleaning.
# Phase 5 is the authoritative cleaned/validated source.
# ============================================================


# ============================================================
# 0. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import mysql.connector

from pathlib import Path
from datetime import datetime
import hashlib
import time
import os

# ============================================================
# 1. CONFIGURATION
# ============================================================

PHASE5_PATH = Path(
    r"C:\Users\hp\Downloads\Working_Capital_ETL\Working_capital_ETL_output.csv"
)

OUTPUT_DIR = PHASE5_PATH.parent

MYSQL_CONFIG = {
    "host": os.getenv("MYSQL_HOST", "localhost"),
    "user": os.getenv("MYSQL_USER", "root"),
    "password": os.getenv("MYSQL_PASSWORD"),
    "database": os.getenv("MYSQL_DATABASE", "working_capital")
}


# ============================================================
# 2. EXPECTED PHASE 5 VALUES
# ============================================================

EXPECTED_PHASE5_ROWS = 12000
EXPECTED_PHASE5_COLUMNS = 52

EXPECTED_SHA256 = (
    "97254dfc20b886e66a3bb57ba4a7ab856b653f1924cd9b26a334b54f557130b6"
)


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def calculate_sha256(file_path):
    """
    Calculate SHA-256 hash of a file.
    """

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


def mysql_safe_value(value):
    """
    Convert Pandas / NumPy scalar values into native Python
    values that mysql.connector can process.

    Examples:

        np.int64(5)       -> Python int
        np.float64(5.5)   -> Python float
        np.bool_(True)    -> Python bool
        pd.NA             -> None
        np.nan            -> None
        pd.Timestamp      -> Python datetime
    """

    if value is None:
        return None

    # Handle Pandas / NumPy missing values
    try:

        if pd.isna(value):
            return None

    except (TypeError, ValueError):

        pass

    # NumPy integers
    if isinstance(value, np.integer):
        return int(value)

    # NumPy floating point
    if isinstance(value, np.floating):
        return float(value)

    # NumPy booleans
    if isinstance(value, np.bool_):
        return bool(value)

    # NumPy strings
    if isinstance(value, np.str_):
        return str(value)

    # Pandas Timestamp
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()

    # NumPy datetime
    if isinstance(value, np.datetime64):
        return pd.Timestamp(value).to_pydatetime()

    return value


def dataframe_to_mysql_rows(df):
    """
    Convert every DataFrame scalar into a native Python
    value before sending it to MySQL.
    """

    rows = []

    for row in df.itertuples(
        index=False,
        name=None
    ):

        safe_row = tuple(
            mysql_safe_value(value)
            for value in row
        )

        rows.append(safe_row)

    return rows


def execute_many_safe(
    cursor,
    sql,
    df,
    batch_size=1000
):
    """
    Safely insert a DataFrame into MySQL in batches.
    """

    rows = dataframe_to_mysql_rows(df)

    total = len(rows)

    for start in range(
        0,
        total,
        batch_size
    ):

        batch = rows[
            start:start + batch_size
        ]

        cursor.executemany(
            sql,
            batch
        )


def table_count(
    cursor,
    schema_name,
    table_name
):
    """
    Actual table row count using COUNT(*).

    We deliberately do NOT use
    INFORMATION_SCHEMA.TABLES.TABLE_ROWS,
    because that can be an estimate.
    """

    sql = f"""
        SELECT COUNT(*)
        FROM `{schema_name}`.`{table_name}`
    """

    cursor.execute(sql)

    result = cursor.fetchone()

    return int(result[0])


def fetch_count_by_column(
    cursor,
    schema_name,
    table_name,
    column_name
):
    """
    Return grouped counts by a column.
    """

    sql = f"""
        SELECT
            `{column_name}`,
            COUNT(*)
        FROM `{schema_name}`.`{table_name}`
        GROUP BY `{column_name}`
        ORDER BY `{column_name}`
    """

    cursor.execute(sql)

    return cursor.fetchall()


def normalize_nulls(df):
    """
    Replace standard missing values with None where possible.

    Nullable Pandas dtypes may still contain pd.NA.
    mysql_safe_value() handles those safely.
    """

    return df.where(
        pd.notna(df),
        None
    )


# ============================================================
# 4. START PHASE 6
# ============================================================

phase_start = time.time()

run_timestamp = datetime.now()

run_id = None

print("=" * 70)
print("PHASE 6 — MYSQL DATA ARCHITECTURE & LOADING")
print("=" * 70)


# ============================================================
# 5. LOAD PHASE 5 OUTPUT
# ============================================================

print("\n1. LOAD PHASE 5 OUTPUT")

if not PHASE5_PATH.exists():

    raise FileNotFoundError(
        f"Phase 5 output file not found:\n{PHASE5_PATH}"
    )

df = pd.read_csv(
    PHASE5_PATH
)

print(
    f"Rows loaded : {len(df):,}"
)

print(
    f"Columns     : {len(df.columns)}"
)


# ============================================================
# 6. VALIDATE PHASE 5 SOURCE
# ============================================================

if len(df) != EXPECTED_PHASE5_ROWS:

    raise ValueError(
        f"Unexpected Phase 5 row count. "
        f"Expected {EXPECTED_PHASE5_ROWS:,}, "
        f"got {len(df):,}"
    )


if len(df.columns) != EXPECTED_PHASE5_COLUMNS:

    raise ValueError(
        f"Unexpected Phase 5 column count. "
        f"Expected {EXPECTED_PHASE5_COLUMNS}, "
        f"got {len(df.columns)}"
    )


if "invoice_id" not in df.columns:

    raise ValueError(
        "invoice_id column is missing."
    )


if df["invoice_id"].isna().any():

    raise ValueError(
        "invoice_id contains missing values."
    )


if not df["invoice_id"].is_unique:

    raise ValueError(
        "invoice_id is not unique in Phase 5 output."
    )


print(
    "Phase 5 source validation: PASS"
)


# ============================================================
# 7. PREPARE MYSQL DATA TYPES
# ============================================================

print("\n2. PREPARE MYSQL DATA TYPES")


# ------------------------------------------------------------
# DATE COLUMNS
# ------------------------------------------------------------

date_columns = [
    "invoice_date",
    "due_date",
    "payment_date"
]

for col in date_columns:

    if col in df.columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        ).dt.date

print(
    "Date conversion     : PASS"
)


# ------------------------------------------------------------
# NUMERIC COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    "invoice_amount",
    "payment_amount",
    "days_to_payment",
    "days_from_due_date",
    "early_payment_days",
    "late_payment_days",
    "invoice_year",
    "invoice_month",
    "quality_issue_count"
]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

print(
    "Numeric conversion  : PASS"
)


# ------------------------------------------------------------
# INTEGER COLUMNS
# ------------------------------------------------------------

integer_columns = [
    "invoice_year",
    "invoice_month",
    "days_to_payment",
    "days_from_due_date",
    "early_payment_days",
    "late_payment_days",
    "quality_issue_count"
]

for col in integer_columns:

    if col in df.columns:

        df[col] = df[col].astype(
            "Int64"
        )

print(
    "Integer conversion  : PASS"
)


# ------------------------------------------------------------
# BOOLEAN COLUMNS
# ------------------------------------------------------------

boolean_columns = [
    col
    for col in df.columns
    if col.startswith("flag_")
]

for col in boolean_columns:

    df[col] = df[col].astype(
        "boolean"
    )

print(
    "Boolean conversion  : PASS"
)


# ------------------------------------------------------------
# NULL NORMALIZATION
# ------------------------------------------------------------

df = normalize_nulls(df)

print(
    "NULL normalization  : PASS"
)


# ============================================================
# 8. SOURCE FILE INTEGRITY
# ============================================================

print("\n3. SOURCE FILE INTEGRITY")

source_sha256 = calculate_sha256(
    PHASE5_PATH
)

print(
    f"SHA-256: {source_sha256}"
)


if source_sha256 != EXPECTED_SHA256:

    raise ValueError(
        "SHA-256 validation failed. "
        "The Phase 5 output is not the expected frozen version."
    )


print(
    "SHA-256 validation: PASS"
)


# ============================================================
# 9. MYSQL CONNECTION
# ============================================================

print("\n4. MYSQL CONNECTION")

connection = mysql.connector.connect(
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    user=MYSQL_USER,
    password=MYSQL_PASSWORD
)

cursor = connection.cursor()

print(
    "MYSQL connection: PASS"
)


# ============================================================
# 10. DATABASE
# ============================================================

print("\n5. DATABASE")

cursor.execute(
    f"""
    CREATE DATABASE IF NOT EXISTS
    `{DATABASE_NAME}`
    """
)

cursor.execute(
    f"""
    USE `{DATABASE_NAME}`
    """
)

print(
    f"Database ready: {DATABASE_NAME}"
)


# ============================================================
# 11. CREATE SCHEMAS
# ============================================================

print("\n6. CREATE SCHEMAS")

schemas = [
    "raw",
    "staging",
    "core",
    "analytics",
    "metadata"
]

for schema in schemas:

    cursor.execute(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        `{schema}`
        """
    )

print(
    "Schemas created: "
    "raw, staging, core, analytics, metadata"
)


# ============================================================
# 12. RESET PHASE 6 TABLES
# ============================================================

print("\n7. RESET PHASE 6 TABLES")

tables_to_drop = [

    ("analytics", "fact_working_capital"),

    ("core", "fact_invoice"),

    ("core", "dim_customer"),

    ("core", "dim_supplier"),

    ("core", "dim_product"),

    ("staging", "working_capital"),

    ("raw", "working_capital"),

    ("metadata", "pipeline_runs")
]

for schema, table in tables_to_drop:

    cursor.execute(
        f"""
        DROP TABLE IF EXISTS
        `{schema}`.`{table}`
        """
    )

connection.commit()

print(
    "Reset: PASS"
)


# ============================================================
# 13. METADATA TABLE
# ============================================================

print("\n8. CREATE METADATA")

cursor.execute("""
    CREATE TABLE metadata.pipeline_runs (

        run_id INT AUTO_INCREMENT PRIMARY KEY,

        pipeline_name VARCHAR(100) NOT NULL,

        source_file VARCHAR(500) NOT NULL,

        source_sha256 CHAR(64) NOT NULL,

        started_at DATETIME NOT NULL,

        finished_at DATETIME NULL,

        status VARCHAR(30) NOT NULL,

        rows_loaded INT NULL,

        error_message TEXT NULL

    )
""")

connection.commit()


cursor.execute(
    """
    INSERT INTO metadata.pipeline_runs
    (
        pipeline_name,
        source_file,
        source_sha256,
        started_at,
        status
    )
    VALUES
    (%s, %s, %s, %s, %s)
    """,
    (
        "working_capital_phase6",
        str(PHASE5_PATH),
        source_sha256,
        run_timestamp,
        "RUNNING"
    )
)

run_id = cursor.lastrowid

connection.commit()

print(
    f"Metadata table created. Run ID: {run_id}"
)


# ============================================================
# 14. RAW TABLE
# ============================================================

print("\n9. CREATE RAW TABLE")

cursor.execute("""
    CREATE TABLE raw.working_capital (

        invoice_id VARCHAR(50),

        customer_id VARCHAR(50),

        customer_name VARCHAR(255),

        region VARCHAR(100),

        supplier_id VARCHAR(50),

        product_id VARCHAR(50),

        invoice_date DATE,

        due_date DATE,

        payment_date DATE,

        invoice_amount DECIMAL(18,2),

        payment_amount DECIMAL(18,2),

        currency VARCHAR(10),

        payment_method VARCHAR(100),

        status VARCHAR(50),

        source_file VARCHAR(500),

        loaded_at DATETIME,

        source_sha256 CHAR(64)

    )
""")

connection.commit()

print(
    "Raw table created"
)


# ============================================================
# 15. RAW DATA PREPARATION
# ============================================================

raw_columns = [

    "invoice_id",
    "customer_id",
    "customer_name",
    "region",
    "supplier_id",
    "product_id",
    "invoice_date",
    "due_date",
    "payment_date",
    "invoice_amount",
    "payment_amount",
    "currency",
    "payment_method",
    "status"
]

raw_df = df[
    raw_columns
].copy()


raw_df["source_file"] = str(
    PHASE5_PATH
)

raw_df["loaded_at"] = run_timestamp

raw_df["source_sha256"] = (
    source_sha256
)


raw_columns_final = raw_columns + [
    "source_file",
    "loaded_at",
    "source_sha256"
]

raw_df = raw_df[
    raw_columns_final
]


# ============================================================
# 16. RAW INSERT
# ============================================================

raw_placeholders = ", ".join(
    ["%s"] * len(raw_columns_final)
)

raw_sql = f"""
    INSERT INTO raw.working_capital
    (
        {
            ", ".join(
                f"`{col}`"
                for col in raw_columns_final
            )
        }
    )
    VALUES
    ({raw_placeholders})
"""


execute_many_safe(
    cursor,
    raw_sql,
    raw_df
)

connection.commit()


raw_count = table_count(
    cursor,
    "raw",
    "working_capital"
)

print(
    f"Raw rows inserted: {raw_count:,}"
)


if raw_count != EXPECTED_PHASE5_ROWS:

    raise ValueError(
        "Raw row-count reconciliation failed."
    )


print(
    "Raw row-count validation: PASS"
)


# ============================================================
# 17. STAGING TABLE
# ============================================================

print("\n10. CREATE STAGING TABLE")


staging_column_sql = []


for col in df.columns:

    # --------------------------------------------------------
    # TEXT / IDENTIFIER COLUMNS
    # --------------------------------------------------------

    if col in [

        "invoice_id",
        "customer_id",
        "customer_name",
        "region",
        "supplier_id",
        "product_id",
        "currency",
        "payment_method",
        "status",
        "original_status",
        "invoice_year_month",
        "business_rule_reason",
        "business_rule_status",
        "financial_analytical_eligibility",
        "quality_status",
        "reconciliation_status"

    ]:

        staging_column_sql.append(
            f"`{col}` VARCHAR(255)"
        )


    # --------------------------------------------------------
    # DATE COLUMNS
    # --------------------------------------------------------

    elif col in [

        "invoice_date",
        "due_date",
        "payment_date"

    ]:

        staging_column_sql.append(
            f"`{col}` DATE"
        )


    # --------------------------------------------------------
    # FINANCIAL COLUMNS
    # --------------------------------------------------------

    elif col in [

        "invoice_amount",
        "payment_amount"

    ]:

        staging_column_sql.append(
            f"`{col}` DECIMAL(18,2)"
        )


    # --------------------------------------------------------
    # INTEGER COLUMNS
    # --------------------------------------------------------

    elif col in [

        "days_to_payment",
        "days_from_due_date",
        "early_payment_days",
        "late_payment_days",
        "invoice_year",
        "invoice_month",
        "quality_issue_count"

    ]:

        staging_column_sql.append(
            f"`{col}` INT"
        )


    # --------------------------------------------------------
    # BOOLEAN FLAGS
    # --------------------------------------------------------

    elif col.startswith("flag_"):

        staging_column_sql.append(
            f"`{col}` BOOLEAN"
        )


    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------

    else:

        staging_column_sql.append(
            f"`{col}` VARCHAR(255)"
        )


staging_column_sql_text = ",\n        ".join(
    staging_column_sql
)


cursor.execute(
    f"""
    CREATE TABLE staging.working_capital (

        {staging_column_sql_text},

        pipeline_run_id INT NOT NULL,

        loaded_at DATETIME NOT NULL,

        INDEX idx_staging_invoice_id
            (invoice_id),

        INDEX idx_staging_status
            (status),

        INDEX idx_staging_customer_id
            (customer_id),

        INDEX idx_staging_supplier_id
            (supplier_id),

        INDEX idx_staging_product_id
            (product_id),

        CONSTRAINT fk_staging_pipeline_run

            FOREIGN KEY (pipeline_run_id)

            REFERENCES metadata.pipeline_runs(run_id)

    )
    """
)

connection.commit()

print(
    "Staging table created"
)


# ============================================================
# 18. STAGING DATA PREPARATION
# ============================================================

staging_df = df.copy()

staging_df["pipeline_run_id"] = run_id

staging_df["loaded_at"] = run_timestamp


staging_columns = list(
    df.columns
) + [
    "pipeline_run_id",
    "loaded_at"
]


staging_df = staging_df[
    staging_columns
]


# ============================================================
# 19. STAGING INSERT
# ============================================================

print(
    "Loading staging rows..."
)


staging_placeholders = ", ".join(
    ["%s"] * len(staging_columns)
)


staging_sql = f"""
    INSERT INTO staging.working_capital
    (
        {
            ", ".join(
                f"`{col}`"
                for col in staging_columns
            )
        }
    )
    VALUES
    ({staging_placeholders})
"""


execute_many_safe(
    cursor,
    staging_sql,
    staging_df
)

connection.commit()


staging_count = table_count(
    cursor,
    "staging",
    "working_capital"
)


print(
    f"Staging rows inserted: {staging_count:,}"
)


if staging_count != EXPECTED_PHASE5_ROWS:

    raise ValueError(
        "Staging row-count reconciliation failed."
    )


print(
    "Staging row-count validation: PASS"
)


# ============================================================
# 20. CUSTOMER DIMENSION
# ============================================================

print("\n11. CREATE CORE DIMENSIONS")


cursor.execute("""
    CREATE TABLE core.dim_customer (

        customer_key INT AUTO_INCREMENT PRIMARY KEY,

        customer_id VARCHAR(50),

        customer_name VARCHAR(255),

        region VARCHAR(100),

        INDEX idx_customer_id
            (customer_id)

    )
""")


cursor.execute("""
    INSERT INTO core.dim_customer
    (
        customer_id,
        customer_name,
        region
    )

    SELECT DISTINCT

        customer_id,
        customer_name,
        region

    FROM staging.working_capital
""")


connection.commit()


customer_count = table_count(
    cursor,
    "core",
    "dim_customer"
)


print(
    f"Customer dimension rows: "
    f"{customer_count:,}"
)


# ============================================================
# 21. SUPPLIER DIMENSION
# ============================================================

cursor.execute("""
    CREATE TABLE core.dim_supplier (

        supplier_key INT AUTO_INCREMENT PRIMARY KEY,

        supplier_id VARCHAR(50) NOT NULL,

        UNIQUE KEY uq_supplier_id
            (supplier_id)

    )
""")


cursor.execute("""
    INSERT INTO core.dim_supplier
    (
        supplier_id
    )

    SELECT DISTINCT

        supplier_id

    FROM staging.working_capital

    WHERE supplier_id IS NOT NULL
""")


connection.commit()


supplier_count = table_count(
    cursor,
    "core",
    "dim_supplier"
)


print(
    f"Supplier dimension rows: "
    f"{supplier_count:,}"
)


# ============================================================
# 22. PRODUCT DIMENSION
# ============================================================

cursor.execute("""
    CREATE TABLE core.dim_product (

        product_key INT AUTO_INCREMENT PRIMARY KEY,

        product_id VARCHAR(50) NOT NULL,

        UNIQUE KEY uq_product_id
            (product_id)

    )
""")


cursor.execute("""
    INSERT INTO core.dim_product
    (
        product_id
    )

    SELECT DISTINCT

        product_id

    FROM staging.working_capital

    WHERE product_id IS NOT NULL
""")


connection.commit()


product_count = table_count(
    cursor,
    "core",
    "dim_product"
)


print(
    f"Product dimension rows: "
    f"{product_count:,}"
)


# ============================================================
# 23. CORE FACT TABLE
# ============================================================

print("\n12. CREATE CORE FACT")


cursor.execute("""
    CREATE TABLE core.fact_invoice (

        invoice_id VARCHAR(50) PRIMARY KEY,

        customer_id VARCHAR(50),

        supplier_id VARCHAR(50),

        product_id VARCHAR(50),

        invoice_date DATE,

        due_date DATE,

        payment_date DATE,

        invoice_amount DECIMAL(18,2),

        payment_amount DECIMAL(18,2),

        currency VARCHAR(10),

        payment_method VARCHAR(100),

        status VARCHAR(50),

        business_rule_status VARCHAR(50),

        financial_analytical_eligibility
            VARCHAR(100),

        days_to_payment INT,

        days_from_due_date INT,

        INDEX idx_fact_customer
            (customer_id),

        INDEX idx_fact_supplier
            (supplier_id),

        INDEX idx_fact_product
            (product_id),

        INDEX idx_fact_invoice_date
            (invoice_date),

        INDEX idx_fact_status
            (status),

        INDEX idx_fact_currency
            (currency),

        CONSTRAINT fk_fact_supplier

            FOREIGN KEY (supplier_id)

            REFERENCES core.dim_supplier(
                supplier_id
            ),

        CONSTRAINT fk_fact_product

            FOREIGN KEY (product_id)

            REFERENCES core.dim_product(
                product_id
            )

    )
""")


cursor.execute("""
    INSERT INTO core.fact_invoice
    (
        invoice_id,
        customer_id,
        supplier_id,
        product_id,
        invoice_date,
        due_date,
        payment_date,
        invoice_amount,
        payment_amount,
        currency,
        payment_method,
        status,
        business_rule_status,
        financial_analytical_eligibility,
        days_to_payment,
        days_from_due_date
    )

    SELECT

        invoice_id,
        customer_id,
        supplier_id,
        product_id,
        invoice_date,
        due_date,
        payment_date,
        invoice_amount,
        payment_amount,
        currency,
        payment_method,
        status,
        business_rule_status,
        financial_analytical_eligibility,
        days_to_payment,
        days_from_due_date

    FROM staging.working_capital
""")


connection.commit()


core_fact_count = table_count(
    cursor,
    "core",
    "fact_invoice"
)


print(
    f"Core fact rows: "
    f"{core_fact_count:,}"
)


# ============================================================
# 24. ANALYTICS FACT TABLE
# ============================================================

print("\n13. CREATE ANALYTICS TABLE")


cursor.execute("""
    CREATE TABLE analytics.fact_working_capital (

        invoice_id VARCHAR(50) PRIMARY KEY,

        customer_id VARCHAR(50),

        supplier_id VARCHAR(50),

        product_id VARCHAR(50),

        invoice_date DATE,

        due_date DATE,

        payment_date DATE,

        invoice_amount DECIMAL(18,2),

        payment_amount DECIMAL(18,2),

        currency VARCHAR(10),

        payment_method VARCHAR(100),

        status VARCHAR(50),

        business_rule_status VARCHAR(50),

        financial_analytical_eligibility
            VARCHAR(100),

        days_to_payment INT,

        days_from_due_date INT,

        INDEX idx_analytics_invoice_date
            (invoice_date),

        INDEX idx_analytics_customer
            (customer_id),

        INDEX idx_analytics_supplier
            (supplier_id),

        INDEX idx_analytics_product
            (product_id),

        INDEX idx_analytics_status
            (status),

        INDEX idx_analytics_currency
            (currency)

    )
""")


# ------------------------------------------------------------
# IMPORTANT:
#
# The target has 16 columns.
# The SELECT below ALSO has exactly 16 columns.
#
# payment_method is explicitly included.
# ------------------------------------------------------------

cursor.execute("""
    INSERT INTO analytics.fact_working_capital
    (
        invoice_id,
        customer_id,
        supplier_id,
        product_id,
        invoice_date,
        due_date,
        payment_date,
        invoice_amount,
        payment_amount,
        currency,
        payment_method,
        status,
        business_rule_status,
        financial_analytical_eligibility,
        days_to_payment,
        days_from_due_date
    )

    SELECT

        invoice_id,
        customer_id,
        supplier_id,
        product_id,
        invoice_date,
        due_date,
        payment_date,
        invoice_amount,
        payment_amount,
        currency,
        payment_method,
        status,
        business_rule_status,
        financial_analytical_eligibility,
        days_to_payment,
        days_from_due_date

    FROM core.fact_invoice
""")


connection.commit()


analytics_count = table_count(
    cursor,
    "analytics",
    "fact_working_capital"
)


print(
    f"Analytics rows: "
    f"{analytics_count:,}"
)


# ============================================================
# 25. ROW COUNT RECONCILIATION
# ============================================================

print("\n14. ROW COUNT RECONCILIATION")


row_count_results = {

    "Phase5":
        len(df),

    "raw.working_capital":
        raw_count,

    "staging.working_capital":
        staging_count,

    "core.fact_invoice":
        core_fact_count,

    "analytics.fact_working_capital":
        analytics_count

}


for layer, count in row_count_results.items():

    expected = len(df)

    status = (
        "PASS"
        if count == expected
        else "FAIL"
    )

    print(
        f"{layer:<35}"
        f"{count:>8,} "
        f"{status}"
    )

    if status == "FAIL":

        raise ValueError(
            f"Row count reconciliation failed "
            f"for {layer}."
        )


# ============================================================
# 26. INVOICE ID RECONCILIATION
# ============================================================

print("\n15. INVOICE ID RECONCILIATION")


phase5_unique_invoice_ids = int(
    df["invoice_id"].nunique()
)


cursor.execute("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM staging.working_capital
""")

staging_unique_invoice_ids = int(
    cursor.fetchone()[0]
)


cursor.execute("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM core.fact_invoice
""")

core_unique_invoice_ids = int(
    cursor.fetchone()[0]
)


cursor.execute("""
    SELECT COUNT(DISTINCT invoice_id)
    FROM analytics.fact_working_capital
""")

analytics_unique_invoice_ids = int(
    cursor.fetchone()[0]
)


invoice_id_results = {

    "Phase5":
        phase5_unique_invoice_ids,

    "Staging":
        staging_unique_invoice_ids,

    "Core":
        core_unique_invoice_ids,

    "Analytics":
        analytics_unique_invoice_ids

}


for layer, count in invoice_id_results.items():

    status = (
        "PASS"
        if count == phase5_unique_invoice_ids
        else "FAIL"
    )

    print(
        f"{layer:<15}"
        f"{count:>8,} "
        f"{status}"
    )

    if status == "FAIL":

        raise ValueError(
            f"Invoice ID reconciliation failed "
            f"for {layer}."
        )


# ============================================================
# 27. REFERENTIAL INTEGRITY
# ============================================================

print("\n16. REFERENTIAL INTEGRITY")


# ------------------------------------------------------------
# SUPPLIER
# ------------------------------------------------------------

cursor.execute("""
    SELECT COUNT(*)

    FROM core.fact_invoice f

    LEFT JOIN core.dim_supplier s

        ON f.supplier_id = s.supplier_id

    WHERE f.supplier_id IS NOT NULL

      AND s.supplier_id IS NULL
""")


missing_supplier_refs = int(
    cursor.fetchone()[0]
)


supplier_status = (
    "PASS"
    if missing_supplier_refs == 0
    else "FAIL"
)


print(
    f"Missing supplier references: "
    f"{missing_supplier_refs} "
    f"{supplier_status}"
)


if missing_supplier_refs != 0:

    raise ValueError(
        "Supplier referential integrity failed."
    )


# ------------------------------------------------------------
# PRODUCT
# ------------------------------------------------------------

cursor.execute("""
    SELECT COUNT(*)

    FROM core.fact_invoice f

    LEFT JOIN core.dim_product p

        ON f.product_id = p.product_id

    WHERE f.product_id IS NOT NULL

      AND p.product_id IS NULL
""")


missing_product_refs = int(
    cursor.fetchone()[0]
)


product_status = (
    "PASS"
    if missing_product_refs == 0
    else "FAIL"
)


print(
    f"Missing product references:  "
    f"{missing_product_refs} "
    f"{product_status}"
)


if missing_product_refs != 0:

    raise ValueError(
        "Product referential integrity failed."
    )


# ============================================================
# 28. BUSINESS KEY RECONCILIATION
# ============================================================

print("\n17. BUSINESS KEY RECONCILIATION")


cursor.execute("""
    SELECT COUNT(DISTINCT supplier_id)
    FROM core.fact_invoice
""")


fact_supplier_keys = int(
    cursor.fetchone()[0]
)


cursor.execute("""
    SELECT COUNT(*)
    FROM core.dim_supplier
""")


dim_supplier_keys = int(
    cursor.fetchone()[0]
)


cursor.execute("""
    SELECT COUNT(DISTINCT product_id)
    FROM core.fact_invoice
""")


fact_product_keys = int(
    cursor.fetchone()[0]
)


cursor.execute("""
    SELECT COUNT(*)
    FROM core.dim_product
""")


dim_product_keys = int(
    cursor.fetchone()[0]
)


print(
    f"Supplier fact keys     : "
    f"{fact_supplier_keys}"
)

print(
    f"Supplier dimension keys: "
    f"{dim_supplier_keys}"
)

print(
    f"Product fact keys      : "
    f"{fact_product_keys}"
)

print(
    f"Product dimension keys : "
    f"{dim_product_keys}"
)


if fact_supplier_keys != dim_supplier_keys:

    raise ValueError(
        "Supplier business-key reconciliation failed."
    )


if fact_product_keys != dim_product_keys:

    raise ValueError(
        "Product business-key reconciliation failed."
    )


print(
    "Business key reconciliation: PASS"
)


# ============================================================
# 29. PHASE 5 → MYSQL INVOICE ID RECONCILIATION
# ============================================================

print(
    "\n18. PHASE 5 → MYSQL INVOICE ID RECONCILIATION"
)


phase5_invoice_ids = set(
    df["invoice_id"]
    .astype(str)
)


cursor.execute("""
    SELECT invoice_id
    FROM core.fact_invoice
""")


mysql_invoice_ids = {

    str(row[0])

    for row in cursor.fetchall()

}


missing_in_mysql = (
    phase5_invoice_ids
    - mysql_invoice_ids
)


extra_in_mysql = (
    mysql_invoice_ids
    - phase5_invoice_ids
)


print(
    f"Missing in MySQL : "
    f"{len(missing_in_mysql)}"
)

print(
    f"Extra in MySQL   : "
    f"{len(extra_in_mysql)}"
)


if len(missing_in_mysql) != 0:

    raise ValueError(
        "Some Phase 5 invoice IDs are "
        "missing from MySQL."
    )


if len(extra_in_mysql) != 0:

    raise ValueError(
        "MySQL contains invoice IDs not "
        "present in Phase 5."
    )


print(
    "Invoice ID reconciliation: PASS"
)


# ============================================================
# 30. FINANCIAL RECONCILIATION
# ============================================================

print("\n19. FINANCIAL AMOUNT RECONCILIATION")


phase5_invoice_total = (
    pd.to_numeric(
        df["invoice_amount"],
        errors="coerce"
    ).sum()
)


phase5_payment_total = (
    pd.to_numeric(
        df["payment_amount"],
        errors="coerce"
    ).sum()
)


cursor.execute("""
    SELECT

        COALESCE(
            SUM(invoice_amount),
            0
        ),

        COALESCE(
            SUM(payment_amount),
            0
        )

    FROM core.fact_invoice
""")


mysql_invoice_total, mysql_payment_total = (
    cursor.fetchone()
)


mysql_invoice_total = float(
    mysql_invoice_total
)

mysql_payment_total = float(
    mysql_payment_total
)


invoice_difference = (
    float(phase5_invoice_total)
    - mysql_invoice_total
)


payment_difference = (
    float(phase5_payment_total)
    - mysql_payment_total
)


print(
    f"Phase5 invoice total : "
    f"{phase5_invoice_total:,.2f}"
)

print(
    f"MySQL invoice total  : "
    f"{mysql_invoice_total:,.2f}"
)

print(
    f"Difference            : "
    f"{invoice_difference:,.2f}"
)


print()


print(
    f"Phase5 payment total : "
    f"{phase5_payment_total:,.2f}"
)

print(
    f"MySQL payment total  : "
    f"{mysql_payment_total:,.2f}"
)

print(
    f"Difference            : "
    f"{payment_difference:,.2f}"
)


if abs(invoice_difference) > 0.01:

    raise ValueError(
        "Invoice amount reconciliation failed."
    )


if abs(payment_difference) > 0.01:

    raise ValueError(
        "Payment amount reconciliation failed."
    )


print(
    "Financial reconciliation: PASS"
)


# ============================================================
# 31. STATUS RECONCILIATION
# ============================================================

print("\n20. STATUS RECONCILIATION")


phase5_status_counts = (
    df["status"]
    .fillna("NULL")
    .value_counts()
    .to_dict()
)


mysql_status_counts = dict(
    fetch_count_by_column(
        cursor,
        "core",
        "fact_invoice",
        "status"
    )
)


mysql_status_counts = {

    (
        "NULL"
        if key is None
        else key
    ): value

    for key, value
    in mysql_status_counts.items()

}


print(
    "Phase 5 status counts:"
)

print(
    phase5_status_counts
)


print(
    "\nMySQL status counts:"
)

print(
    mysql_status_counts
)


if phase5_status_counts != mysql_status_counts:

    raise ValueError(
        "Status reconciliation failed."
    )


print(
    "Status reconciliation: PASS"
)


# ============================================================
# 32. CURRENCY RECONCILIATION
# ============================================================

print("\n21. CURRENCY RECONCILIATION")


phase5_currency_counts = (
    df["currency"]
    .fillna("NULL")
    .value_counts()
    .to_dict()
)


mysql_currency_counts = dict(
    fetch_count_by_column(
        cursor,
        "core",
        "fact_invoice",
        "currency"
    )
)


mysql_currency_counts = {

    (
        "NULL"
        if key is None
        else key
    ): value

    for key, value
    in mysql_currency_counts.items()

}


print(
    "Phase 5 currency counts:"
)

print(
    phase5_currency_counts
)


print(
    "\nMySQL currency counts:"
)

print(
    mysql_currency_counts
)


if phase5_currency_counts != mysql_currency_counts:

    raise ValueError(
        "Currency reconciliation failed."
    )


print(
    "Currency reconciliation: PASS"
)


# ============================================================
# 33. DATA TYPE INSPECTION
# ============================================================

print("\n22. DATA TYPE INSPECTION")


cursor.execute("""
    SELECT

        TABLE_SCHEMA,

        TABLE_NAME,

        COLUMN_NAME,

        DATA_TYPE,

        COLUMN_TYPE

    FROM information_schema.COLUMNS

    WHERE TABLE_SCHEMA IN
        (
            'raw',
            'staging',
            'core',
            'analytics'
        )

    ORDER BY

        TABLE_SCHEMA,

        TABLE_NAME,

        ORDINAL_POSITION
""")


type_rows = cursor.fetchall()


print(
    f"Inspected "
    f"{len(type_rows):,} "
    f"MySQL column definitions."
)


print(
    "Data type inspection: PASS"
)


# ============================================================
# 34. ACTUAL TABLE INVENTORY
# ============================================================

print("\n23. ACTUAL TABLE INVENTORY")


inventory_tables = [

    ("raw", "working_capital"),

    ("staging", "working_capital"),

    ("core", "dim_customer"),

    ("core", "dim_supplier"),

    ("core", "dim_product"),

    ("core", "fact_invoice"),

    ("analytics", "fact_working_capital"),

    ("metadata", "pipeline_runs")

]


inventory = []


for schema, table in inventory_tables:

    count = table_count(
        cursor,
        schema,
        table
    )

    inventory.append({

        "schema": schema,

        "table": table,

        "row_count": count

    })

    print(
        f"{schema:<12}"
        f"{table:<25}"
        f"{count:>8,}"
    )


# ============================================================
# 35. INVENTORY VALIDATION
# ============================================================

print("\n24. INVENTORY VALIDATION")


expected_inventory = {

    ("raw", "working_capital"):
        12000,

    ("staging", "working_capital"):
        12000,

    ("core", "dim_supplier"):
        30,

    ("core", "dim_product"):
        40,

    ("core", "fact_invoice"):
        12000,

    ("analytics", "fact_working_capital"):
        12000,

    ("metadata", "pipeline_runs"):
        1

}


for item in inventory:

    key = (
        item["schema"],
        item["table"]
    )

    actual = item["row_count"]


    if key in expected_inventory:

        expected = expected_inventory[key]

        status = (
            "PASS"
            if actual == expected
            else "FAIL"
        )

        print(
            f"{key[0]}.{key[1]:<25}"
            f"expected={expected:,} "
            f"actual={actual:,} "
            f"{status}"
        )


        if status == "FAIL":

            raise ValueError(
                f"Inventory validation failed "
                f"for {key[0]}.{key[1]}"
            )


print(
    "Inventory validation: PASS"
)


# ============================================================
# 36. FINAL PIPELINE STATUS
# ============================================================

phase_end = time.time()

elapsed_seconds = (
    phase_end
    - phase_start
)


print("\n25. FINAL PIPELINE STATUS")


cursor.execute("""
    UPDATE metadata.pipeline_runs

    SET

        finished_at = %s,

        status = %s,

        rows_loaded = %s

    WHERE run_id = %s
""", (

    datetime.now(),

    "SUCCESS",

    EXPECTED_PHASE5_ROWS,

    run_id

))


connection.commit()


print(
    "Pipeline status: SUCCESS"
)

print(
    f"Rows loaded    : "
    f"{EXPECTED_PHASE5_ROWS:,}"
)

print(
    f"Run ID         : "
    f"{run_id}"
)

print(
    f"Elapsed time   : "
    f"{elapsed_seconds:.2f} seconds"
)


# ============================================================
# 37. SAVE PHASE 6 REPORT
# ============================================================

phase6_report = pd.DataFrame({

    "metric": [

        "phase5_rows",

        "phase5_columns",

        "raw_rows",

        "staging_rows",

        "customer_dimension_rows",

        "supplier_dimension_rows",

        "product_dimension_rows",

        "core_fact_rows",

        "analytics_rows",

        "phase5_invoice_ids",

        "mysql_invoice_ids",

        "phase5_invoice_total",

        "mysql_invoice_total",

        "phase5_payment_total",

        "mysql_payment_total",

        "missing_supplier_refs",

        "missing_product_refs",

        "run_id",

        "source_sha256",

        "status",

        "elapsed_seconds"

    ],

    "value": [

        len(df),

        len(df.columns),

        raw_count,

        staging_count,

        customer_count,

        supplier_count,

        product_count,

        core_fact_count,

        analytics_count,

        phase5_unique_invoice_ids,

        len(mysql_invoice_ids),

        round(
            float(
                phase5_invoice_total
            ),
            2
        ),

        round(
            float(
                mysql_invoice_total
            ),
            2
        ),

        round(
            float(
                phase5_payment_total
            ),
            2
        ),

        round(
            float(
                mysql_payment_total
            ),
            2
        ),

        missing_supplier_refs,

        missing_product_refs,

        run_id,

        source_sha256,

        "SUCCESS",

        round(
            elapsed_seconds,
            2
        )

    ]

})


phase6_report_path = (
    OUTPUT_DIR
    / "Phase6_MySQL_Architecture_Report.csv"
)


phase6_report.to_csv(
    phase6_report_path,
    index=False
)


print(
    "\nPhase 6 report saved:"
)

print(
    phase6_report_path
)


# ============================================================
# 38. CLOSE MYSQL CONNECTION
# ============================================================

cursor.close()

connection.close()


print(
    "\nMySQL connection closed."
)


print(
    "\n" + "=" * 70
)

print(
    "PHASE 6 COMPLETE — SUCCESS"
)

print(
    "=" * 70
)

PHASE 6 — MYSQL DATA ARCHITECTURE & LOADING

1. LOAD PHASE 5 OUTPUT
Rows loaded : 12,000
Columns     : 52
Phase 5 source validation: PASS

2. PREPARE MYSQL DATA TYPES
Date conversion     : PASS
Numeric conversion  : PASS
Integer conversion  : PASS
Boolean conversion  : PASS
NULL normalization  : PASS

3. SOURCE FILE INTEGRITY
SHA-256: 97254dfc20b886e66a3bb57ba4a7ab856b653f1924cd9b26a334b54f557130b6
SHA-256 validation: PASS

4. MYSQL CONNECTION
MYSQL connection: PASS

5. DATABASE
Database ready: working_capital

6. CREATE SCHEMAS
Schemas created: raw, staging, core, analytics, metadata

7. RESET PHASE 6 TABLES
Reset: PASS

8. CREATE METADATA
Metadata table created. Run ID: 1

9. CREATE RAW TABLE
Raw table created
Raw rows inserted: 12,000
Raw row-count validation: PASS

10. CREATE STAGING TABLE
Staging table created
Loading staging rows...
Staging rows inserted: 12,000
Staging row-count validation: PASS

11. CREATE CORE DIMENSIONS
Customer dimension rows: 4,385
Supplier dimension rows: 